# Clothes segmentation: Colab training runner (Part 2)

This notebook only orchestrates. All logic lives in the `.py` files of the project.

**Before running**
1. `Runtime > Change runtime type`: choose **L4 GPU**.
2. Run the cells from top to bottom. Cell 3 asks you to upload the new `clothseg_code.zip` (from the project folder on your computer) once.
3. Every cell is safe to rerun. If the session dies, open the notebook again and run everything from the top: finished work is skipped and training continues from the last checkpoint.

**What it does:** prepares the data, proves that resume works, then runs bake-off round 1 (4 models, 5 epochs each, about 3.5 hours on an L4).

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Eyad-Elnakib1/Atelier.git"  # this repository; set to "" to use an uploaded clothseg_code.zip instead

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"   # everything persistent lives here
CODE_DIR = "/content/clothes-segmentation"       # local copy of the code
CODE_VERSION = "19"                                # version of clothseg_code.zip this notebook needs
HEIGHT, WIDTH = 576, 384                          # training resolution (H, W)
SESSION_BUDGET_MIN = 600                          # training stops cleanly and saves this many minutes after cell 2 ran

MODELS = ["segformer_b2", "segformer_b3", "segformer_b4", "upernet_convnext_tiny"]
SCHEDULE_EPOCHS = 12                              # length of the learning-rate schedule (same for every model)
ROUND1_EPOCHS = 5                                 # round 1 stops here
EXTRA_MODELS = ["deeplabv3plus_resnet101"]        # trained separately in cell 9

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil, time
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
SESSION_START = time.time()

def minutes_left():
    """Minutes of the session budget that remain; passed to every training command."""
    return max(1, SESSION_BUDGET_MIN - (time.time() - SESSION_START) / 60)

import torch, transformers
print("python      ", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > L4 GPU, then run again."

## 3. Get the code

In [ ]:
import subprocess, zipfile

def code_version(folder):
    path = os.path.join(folder, "VERSION")
    return open(path).read().strip() if os.path.exists(path) else "1"

os.chdir("/content")
if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and code_version(f"{DRIVE_ROOT}/code") == CODE_VERSION:
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR)
    print(f"code version {CODE_VERSION} copied from Drive")
else:
    from google.colab import files
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print(f"Drive has code version {code_version(f'{DRIVE_ROOT}/code')}, this notebook needs version {CODE_VERSION}.")
    print("Choose the NEW clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

assert code_version(CODE_DIR) == CODE_VERSION, (
    f"The code is version {code_version(CODE_DIR)} but this notebook needs version {CODE_VERSION}. "
    "You uploaded an old clothseg_code.zip. Run this cell again and choose the new one.")
for needed in ("scripts/train.py", "scripts/resume_test.py", "src/clothseg/engine.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd(), "| version", code_version(CODE_DIR))

## 4. Install requirements

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm, cv2
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| opencv", cv2.__version__)

## 5. Prepare the dataset

Copies the archive from Drive and extracts it (about a minute). It then moves near-duplicate photos that sit in more than one split into train, so validation and test contain no copy of a training image.

In [ ]:
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 6. Resume test (about 5 minutes)

Trains a tiny experiment twice, once without interruption and once killed and resumed twice, then compares them step by step. It must end with **RESUME TEST PASSED** before the long runs start.

In [ ]:
!python scripts/resume_test.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH} 2>&1 | tee /content/resume_test.log
assert "RESUME TEST PASSED" in open("/content/resume_test.log").read(), "The resume test did not pass. Do not start the long runs; send me the output above."

## 7. Bake-off round 1

Trains each model for 5 epochs of a 12-epoch schedule, one after another. Expect roughly 40 to 65 minutes per model.

If the session dies, run the notebook again from the top: finished models are skipped and the interrupted one continues from its last checkpoint.

In [ ]:
for m in MODELS:
    print("\n" + "=" * 100 + f"\n{m}\n" + "=" * 100)
    budget = int(minutes_left())
    !python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name r1_{m} --model {m} --height {HEIGHT} --width {WIDTH} --epochs {SCHEDULE_EPOCHS} --stop-after-epoch {ROUND1_EPOCHS} --time-budget-min {budget}

## 8. Compare the models

In [ ]:
!python scripts/compare_runs.py --drive-root "{DRIVE_ROOT}" --prefix r1_ --at-epoch {ROUND1_EPOCHS}

from IPython.display import Image, display
display(Image(f"{DRIVE_ROOT}/reports/compare_r1.png"))

## What to send back

Copy the full text output of cells 6 and 8, and the `[val]` lines of cell 7. If a cell fails, copy the whole error message.

Cell 9 is optional. **Do not run cell 10** (round 2) until we have picked the top two models together.

## 9. Extra candidate: DeepLabV3+ (optional, about 50 minutes)

Trains DeepLabV3+ (ResNet-101 encoder) with exactly the same settings as the round 1 models, then prints the comparison again with all five models. Run it after cell 8. If you run it, send back its output too.

In [ ]:
for m in EXTRA_MODELS:
    print("\n" + "=" * 100 + f"\n{m}\n" + "=" * 100)
    budget = int(minutes_left())
    !python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name r1_{m} --model {m} --height {HEIGHT} --width {WIDTH} --epochs {SCHEDULE_EPOCHS} --stop-after-epoch {ROUND1_EPOCHS} --time-budget-min {budget}

!python scripts/compare_runs.py --drive-root "{DRIVE_ROOT}" --prefix r1_ --at-epoch {ROUND1_EPOCHS} --out "{DRIVE_ROOT}/reports/compare_r1_with_extra.png"
from IPython.display import Image, display
display(Image(f"{DRIVE_ROOT}/reports/compare_r1_with_extra.png"))

## 10. Bake-off round 2 (only after we choose the top two)

Put the two chosen model names in `TOP2`. The same runs continue from epoch 6 to epoch 12.

In [ ]:
TOP2 = []   # e.g. ["segformer_b3", "segformer_b2"]

assert len(TOP2) == 2 and all(m in MODELS + EXTRA_MODELS for m in TOP2), "Fill TOP2 with the two model names we agreed on."
for m in TOP2:
    print("\n" + "=" * 100 + f"\n{m}\n" + "=" * 100)
    budget = int(minutes_left())
    !python scripts/train.py --drive-root "{DRIVE_ROOT}" --run-name r1_{m} --model {m} --height {HEIGHT} --width {WIDTH} --epochs {SCHEDULE_EPOCHS} --time-budget-min {budget}

!python scripts/compare_runs.py --drive-root "{DRIVE_ROOT}" --prefix r1_ --out "{DRIVE_ROOT}/reports/compare_r1_round2.png"
from IPython.display import Image, display
display(Image(f"{DRIVE_ROOT}/reports/compare_r1_round2.png"))